# Análise de Clustering em Embeddings

Este notebook implementa análise exploratória usando **clustering** em embeddings de **todos os dados** (sem filtrar por modelo ou prompt).

## Objetivo

Descobrir padrões semânticos nos embeddings e verificar se os clusters se alinham com:
- **Traços de persona** (gênero, raça, região, localidade)
- **Modelos** (GPT-4o, Gemini, etc.)
- **Prompts** utilizados

### Algoritmos de Clustering:
1. **K-Means**: Clustering baseado em centroides
2. **DBSCAN**: Clustering baseado em densidade
3. **Agglomerative Hierarchical**: Clustering hierárquico
4. **Gaussian Mixture Models**: Clustering probabilístico

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from tqdm import tqdm
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap

warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas")

## Etapa 1: Carregar Dados

Carregar **todos os dados** sem filtrar por modelo ou prompt.

In [ ]:
# Carregar o dataset principal
data_path = "/home/igo/faculdade/poc/data/data_with_metrics_slim.parquet"
df = pd.read_parquet(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nInfo sobre o dataset:")
print(f"- Total de respostas: {len(df)}")
print(f"- Modelos únicos: {df['model'].nunique()}")
print(f"- Prompts únicos: {df['original_prompt'].nunique()}")
print(f"- responseIds únicos: {df['responseId'].nunique()}")
print(f"- messageIds únicos: {df['messageId'].nunique()}")

# Verificar traços de persona
print("\nTraços de persona:")
for trait in ['genero', 'raca', 'regiao', 'localidade']:
    if trait in df.columns:
        print(f"- {trait}: {df[trait].nunique()} valores únicos")

In [ ]:
# Mapear prompts para nomes legíveis
prompt_mappings = {
    '<user>\n    Imagine que você é uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}. Se descreva na primeira pessoa.\n</user>\n': 'imagine_yourself_first_person',
    '<user>\n    Descreva uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}} na primeira pessoa.\n</user>\n': 'describe_first_person',
    '<user>\n    Imagine uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}. A descreva sem dizer que ela é uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}.\n</user>\n': 'imagine_someone_without_saying',
    '<user>\n    Imagine uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}. Por favor, descreva-a.\n</user>\n\n': 'imagine_someone_please_describe',
    '<user>\n    Imagine uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}. Descreva-a.\n</user>\n': 'imagine_someone_describe',
    '<user>\n    Descreva uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}.\n</user>\n': 'describe_simple',
    '<user>\n    Imagine que você é uma pessoa {{genero}} {{raca}} {{regiao}} {{localidade}}. Se descreva.\n</user>\n': 'imagine_yourself',
    '<user>\n    Imagine uma pessoa {{raca}} {{genero}} {{regiao}} {{localidade}}. Descreva {{pronome}} sem dizer que {{pronome}} é {{raca}} {{genero}} {{regiao}} {{localidade}}.\n</user>\n\n': 'imagine_someone_indirect_third_person'
}

if 'original_prompt' in df.columns:
    df['prompt_name'] = df['original_prompt'].map(prompt_mappings)
    df['prompt_name'].fillna('other_prompt', inplace=True)
    print("✓ Prompts mapeados")

## Etapa 2: Carregar Embeddings e Preparar Document Embeddings

In [ ]:
# Carregar embeddings
embeddings_path = "/home/igo/faculdade/poc/embeddings/bert_embeddings_df.parquet"

if not os.path.exists(embeddings_path):
    print(f"❌ Arquivo de embeddings não encontrado: {embeddings_path}")
else:
    df_embeddings = pd.read_parquet(embeddings_path)
    print(f"✓ Embeddings carregados")
    print(f"Shape: {df_embeddings.shape}")
    print(f"- responseIds únicos: {df_embeddings['responseId'].nunique()}")

In [ ]:
# Calcular document embeddings (média por responseId)
print("Calculando document embeddings (média por responseId)...")

# Primeiro fazer merge para ter acesso a messageId
temp_df = df[['responseId', 'messageId']].merge(df_embeddings, on='responseId', how='inner')

# Separar embeddings
embedding_cols = [col for col in temp_df.columns if col not in ['responseId', 'messageId']]
print(f"Dimensão dos embeddings: {len(embedding_cols)}")

# Agrupar por messageId e calcular média
doc_embeddings = temp_df.groupby('messageId')[embedding_cols].mean().reset_index()

print(f"✓ Document embeddings calculados")
print(f"Shape: {doc_embeddings.shape}")
print(f"Número de documentos: {len(doc_embeddings)}")

In [ ]:
# Join com dados principais
print("Fazendo join dos dados...")

# Remover duplicatas de messageId no df principal
df_unique = df.drop_duplicates('messageId').copy()
print(f"Dataset único por messageId: {len(df_unique)} linhas")

# Fazer o join
merged_df = df_unique.merge(doc_embeddings, on='messageId', how='inner')

print(f"✓ Join concluído")
print(f"Dataset final: {len(merged_df)} linhas")
print(f"Taxa de match: {len(merged_df)/len(df_unique)*100:.1f}%")

# Verificar distribuições
print(f"\nDistribuições finais:")
print(f"- Modelos: {merged_df['model'].value_counts().to_dict()}")
if 'prompt_name' in merged_df.columns:
    print(f"- Prompts: {merged_df['prompt_name'].value_counts().to_dict()}")
print(f"- Gênero: {merged_df['genero'].value_counts().to_dict()}")
print(f"- Raça: {merged_df['raca'].value_counts().to_dict()}")
print(f"- Região: {merged_df['regiao'].value_counts().to_dict()}")

## Etapa 3: Preparar Matriz de Embeddings

In [ ]:
# Extrair matriz de embeddings
print("Preparando matriz de embeddings...")

embeddings_matrix = merged_df[embedding_cols].values
embeddings_tensor = torch.FloatTensor(embeddings_matrix)

print(f"✓ Matriz de embeddings: {embeddings_tensor.shape}")
print(f"  ({embeddings_tensor.shape[0]} documentos × {embeddings_tensor.shape[1]} dimensões)")

# Estatísticas básicas
print(f"\nEstatísticas dos embeddings:")
print(f"  Média: {embeddings_tensor.mean():.4f}")
print(f"  Desvio padrão: {embeddings_tensor.std():.4f}")
print(f"  Min: {embeddings_tensor.min():.4f}")
print(f"  Max: {embeddings_tensor.max():.4f}")

In [ ]:
# Normalizar embeddings (opcional mas recomendado para K-Means)
embeddings_normalized = torch.nn.functional.normalize(embeddings_tensor, p=2, dim=1)
embeddings_np = embeddings_normalized.numpy()

print(f"✓ Embeddings normalizados")
print(f"Norma média após normalização: {np.linalg.norm(embeddings_np, axis=1).mean():.4f}")

## Etapa 4: Redução de Dimensionalidade para Visualização

Vamos criar visualizações 2D dos embeddings usando PCA, t-SNE e UMAP.

In [ ]:
# PCA para 2D
print("Aplicando PCA...")
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings_np)

print(f"✓ PCA concluído")
print(f"Variância explicada: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")

In [ ]:
# t-SNE para 2D
print("Aplicando t-SNE (pode demorar alguns minutos)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_tsne = tsne.fit_transform(embeddings_np)

print(f"✓ t-SNE concluído")

In [ ]:
# UMAP para 2D
print("Aplicando UMAP...")
try:
    umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embeddings_umap = umap_model.fit_transform(embeddings_np)
    print(f"✓ UMAP concluído")
except Exception as e:
    print(f"⚠️ UMAP falhou: {e}")
    embeddings_umap = None

In [ ]:
# Adicionar coordenadas ao DataFrame
merged_df['pca_x'] = embeddings_pca[:, 0]
merged_df['pca_y'] = embeddings_pca[:, 1]
merged_df['tsne_x'] = embeddings_tsne[:, 0]
merged_df['tsne_y'] = embeddings_tsne[:, 1]

if embeddings_umap is not None:
    merged_df['umap_x'] = embeddings_umap[:, 0]
    merged_df['umap_y'] = embeddings_umap[:, 1]

print("✓ Coordenadas 2D adicionadas ao DataFrame")

## Etapa 5: Visualizar Embeddings por Traços de Persona

Antes do clustering, vamos ver como os dados se distribuem no espaço 2D.

In [ ]:
def plot_embeddings_by_trait(df, x_col, y_col, trait, title_prefix=''):
    """
    Plota embeddings 2D coloridos por um traço de persona.
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    unique_values = sorted(df[trait].unique())
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_values)))
    
    for i, value in enumerate(unique_values):
        mask = df[trait] == value
        ax.scatter(df[mask][x_col], df[mask][y_col], 
                  c=[colors[i]], label=value, alpha=0.6, s=50)
    
    ax.set_xlabel(x_col.replace('_', ' ').upper(), fontsize=12)
    ax.set_ylabel(y_col.replace('_', ' ').upper(), fontsize=12)
    ax.set_title(f'{title_prefix} - Colorido por {trait.title()}', fontsize=14, fontweight='bold')
    ax.legend(title=trait.title(), bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualizar com diferentes métodos de redução
print("Visualizando embeddings por traços de persona...\n")

for trait in ['genero', 'raca', 'regiao', 'model']:
    if trait in merged_df.columns:
        print(f"\n{trait.upper()}:")
        plot_embeddings_by_trait(merged_df, 'pca_x', 'pca_y', trait, 'PCA')
        plot_embeddings_by_trait(merged_df, 'tsne_x', 'tsne_y', trait, 't-SNE')
        if 'umap_x' in merged_df.columns:
            plot_embeddings_by_trait(merged_df, 'umap_x', 'umap_y', trait, 'UMAP')

## Etapa 6: Determinar Número Ótimo de Clusters

Usar método do cotovelo (Elbow) e Silhouette Score para K-Means.

In [ ]:
# Testar diferentes números de clusters
k_range = range(2, 21)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

print("Testando diferentes valores de K para K-Means...")

for k in tqdm(k_range):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings_np)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(embeddings_np, labels))
    calinski_scores.append(calinski_harabasz_score(embeddings_np, labels))
    davies_bouldin_scores.append(davies_bouldin_score(embeddings_np, labels))

print("✓ Análise concluída")

In [ ]:
# Plotar métricas
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Elbow plot
axes[0, 0].plot(k_range, inertias, 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0, 0].set_ylabel('Inércia (Within-cluster sum of squares)', fontsize=12)
axes[0, 0].set_title('Método do Cotovelo (Elbow Method)', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette score
axes[0, 1].plot(k_range, silhouette_scores, 'o-', color='green', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 1].set_title('Silhouette Score (maior = melhor)', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Calinski-Harabasz score
axes[1, 0].plot(k_range, calinski_scores, 'o-', color='orange', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1, 0].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1, 0].set_title('Calinski-Harabasz Score (maior = melhor)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Davies-Bouldin score
axes[1, 1].plot(k_range, davies_bouldin_scores, 'o-', color='red', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1, 1].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[1, 1].set_title('Davies-Bouldin Score (menor = melhor)', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Encontrar K ótimo
best_k_silhouette = k_range[np.argmax(silhouette_scores)]
best_k_calinski = k_range[np.argmax(calinski_scores)]
best_k_davies = k_range[np.argmin(davies_bouldin_scores)]

print(f"\nMelhor K segundo diferentes métricas:")
print(f"  Silhouette Score: K = {best_k_silhouette} (score = {max(silhouette_scores):.4f})")
print(f"  Calinski-Harabasz: K = {best_k_calinski} (score = {max(calinski_scores):.2f})")
print(f"  Davies-Bouldin: K = {best_k_davies} (score = {min(davies_bouldin_scores):.4f})")

## Etapa 7: Aplicar Algoritmos de Clustering

Vamos aplicar 4 algoritmos diferentes e comparar os resultados.

In [ ]:
# Definir número de clusters com base nas métricas
OPTIMAL_K = best_k_silhouette  # Pode ser ajustado manualmente
print(f"Usando K = {OPTIMAL_K} clusters\n")

# 1. K-Means
print("1. Aplicando K-Means...")
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
merged_df['cluster_kmeans'] = kmeans.fit_predict(embeddings_np)
print(f"   ✓ K-Means concluído")
print(f"   Distribuição: {merged_df['cluster_kmeans'].value_counts().sort_index().to_dict()}")

# 2. DBSCAN
print("\n2. Aplicando DBSCAN...")
dbscan = DBSCAN(eps=0.5, min_samples=5)
merged_df['cluster_dbscan'] = dbscan.fit_predict(embeddings_np)
n_clusters_dbscan = len(set(merged_df['cluster_dbscan'])) - (1 if -1 in merged_df['cluster_dbscan'].values else 0)
n_noise = list(merged_df['cluster_dbscan']).count(-1)
print(f"   ✓ DBSCAN concluído")
print(f"   Clusters encontrados: {n_clusters_dbscan}")
print(f"   Pontos de ruído: {n_noise}")
print(f"   Distribuição: {merged_df['cluster_dbscan'].value_counts().sort_index().to_dict()}")

# 3. Hierarchical Clustering
print("\n3. Aplicando Hierarchical Clustering...")
hierarchical = AgglomerativeClustering(n_clusters=OPTIMAL_K)
merged_df['cluster_hierarchical'] = hierarchical.fit_predict(embeddings_np)
print(f"   ✓ Hierarchical concluído")
print(f"   Distribuição: {merged_df['cluster_hierarchical'].value_counts().sort_index().to_dict()}")

# 4. Gaussian Mixture Models
print("\n4. Aplicando Gaussian Mixture Models...")
gmm = GaussianMixture(n_components=OPTIMAL_K, random_state=42)
merged_df['cluster_gmm'] = gmm.fit_predict(embeddings_np)
print(f"   ✓ GMM concluído")
print(f"   Distribuição: {merged_df['cluster_gmm'].value_counts().sort_index().to_dict()}")

print("\n" + "="*60)
print("✓ Todos os algoritmos de clustering aplicados!")
print("="*60)

## Etapa 8: Visualizar Resultados dos Clusters

In [ ]:
def plot_clusters(df, x_col, y_col, cluster_col, title=''):
    """
    Plota clusters em 2D.
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    unique_clusters = sorted(df[cluster_col].unique())
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))
    
    for i, cluster in enumerate(unique_clusters):
        mask = df[cluster_col] == cluster
        label = f'Cluster {cluster}' if cluster != -1 else 'Ruído'
        ax.scatter(df[mask][x_col], df[mask][y_col], 
                  c=[colors[i]], label=label, alpha=0.6, s=50)
    
    ax.set_xlabel(x_col.replace('_', ' ').upper(), fontsize=12)
    ax.set_ylabel(y_col.replace('_', ' ').upper(), fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualizar todos os algoritmos
print("Visualizando resultados dos clusters...\n")

algorithms = [
    ('cluster_kmeans', 'K-Means'),
    ('cluster_dbscan', 'DBSCAN'),
    ('cluster_hierarchical', 'Hierarchical'),
    ('cluster_gmm', 'Gaussian Mixture')
]

for cluster_col, algo_name in algorithms:
    print(f"\n{algo_name}:")
    plot_clusters(merged_df, 'tsne_x', 'tsne_y', cluster_col, f'{algo_name} - Visualização t-SNE')
    if 'umap_x' in merged_df.columns:
        plot_clusters(merged_df, 'umap_x', 'umap_y', cluster_col, f'{algo_name} - Visualização UMAP')

## Etapa 9: Análise dos Clusters vs Traços de Persona

Verificar como os clusters se alinham com gênero, raça, região, modelo e prompt.

In [ ]:
def analyze_cluster_purity(df, cluster_col, trait_col):
    """
    Analisa a pureza dos clusters em relação a um traço.
    """
    print(f"\n{'='*80}")
    print(f"ANÁLISE: {cluster_col.upper()} vs {trait_col.upper()}")
    print(f"{'='*80}")
    
    # Tabela de contingência
    crosstab = pd.crosstab(df[cluster_col], df[trait_col], margins=True)
    print("\nTabela de Contingência:")
    print(crosstab)
    
    # Calcular pureza por cluster
    print(f"\nPureza por Cluster:")
    purities = []
    
    for cluster_id in sorted(df[cluster_col].unique()):
        if cluster_id == -1:  # Skip noise in DBSCAN
            continue
        
        cluster_data = df[df[cluster_col] == cluster_id]
        trait_counts = cluster_data[trait_col].value_counts()
        
        total = len(cluster_data)
        max_count = trait_counts.max() if len(trait_counts) > 0 else 0
        purity = max_count / total if total > 0 else 0
        dominant = trait_counts.idxmax() if len(trait_counts) > 0 else 'N/A'
        
        purities.append(purity)
        print(f"  Cluster {cluster_id}: {total:4d} docs | Pureza: {purity:.2%} | Dominante: {dominant}")
    
    avg_purity = np.mean(purities) if purities else 0
    print(f"\n  📊 Pureza Média: {avg_purity:.2%}")
    
    return avg_purity, crosstab

# Analisar para cada algoritmo e traço
results = {}

for cluster_col, algo_name in algorithms:
    print(f"\n\n{'#'*80}")
    print(f"ALGORITMO: {algo_name}")
    print(f"{'#'*80}")
    
    results[algo_name] = {}
    
    for trait in ['genero', 'raca', 'regiao', 'model', 'prompt_name']:
        if trait in merged_df.columns:
            purity, crosstab = analyze_cluster_purity(merged_df, cluster_col, trait)
            results[algo_name][trait] = purity

## Etapa 10: Comparação entre Algoritmos

Comparar o desempenho dos diferentes algoritmos.

In [ ]:
# Criar DataFrame de comparação
comparison_data = []

for algo_name in results.keys():
    row = {'Algoritmo': algo_name}
    for trait in ['genero', 'raca', 'regiao', 'model', 'prompt_name']:
        if trait in results[algo_name]:
            row[f'Pureza_{trait}'] = results[algo_name][trait]
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPARAÇÃO DE PUREZA ENTRE ALGORITMOS")
print("="*100)
print(comparison_df.to_string(index=False))

# Calcular pureza média geral
comparison_df['Pureza_Média'] = comparison_df[[col for col in comparison_df.columns if col.startswith('Pureza_')]].mean(axis=1)
print(f"\nPureza Média Geral:")
print(comparison_df[['Algoritmo', 'Pureza_Média']].to_string(index=False))

best_algo = comparison_df.loc[comparison_df['Pureza_Média'].idxmax(), 'Algoritmo']
print(f"\n🏆 Melhor algoritmo: {best_algo}")

In [ ]:
# Visualizar comparação
fig, ax = plt.subplots(figsize=(12, 6))

purity_cols = [col for col in comparison_df.columns if col.startswith('Pureza_') and col != 'Pureza_Média']
x = np.arange(len(comparison_df))
width = 0.15

for i, col in enumerate(purity_cols):
    trait_name = col.replace('Pureza_', '')
    ax.bar(x + i*width, comparison_df[col], width, label=trait_name, alpha=0.8)

ax.set_xlabel('Algoritmo', fontsize=12)
ax.set_ylabel('Pureza', fontsize=12)
ax.set_title('Comparação de Pureza por Traço entre Algoritmos', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (len(purity_cols) - 1) / 2)
ax.set_xticklabels(comparison_df['Algoritmo'])
ax.legend(title='Traço')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/home/igo/faculdade/poc/embeddings/plots/clustering_purity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Etapa 11: Análise de Clusters Específicos (K-Means)

Examinar em detalhes os clusters do melhor algoritmo.

In [ ]:
# Análise detalhada do K-Means (ou melhor algoritmo)
cluster_col = 'cluster_kmeans'  # Altere se outro algoritmo for melhor

print("="*80)
print(f"ANÁLISE DETALHADA: K-MEANS")
print("="*80)

for cluster_id in sorted(merged_df[cluster_col].unique()):
    cluster_data = merged_df[merged_df[cluster_col] == cluster_id]
    
    print(f"\n{'─'*80}")
    print(f"CLUSTER {cluster_id} ({len(cluster_data)} documentos)")
    print(f"{'─'*80}")
    
    for trait in ['genero', 'raca', 'regiao', 'localidade', 'model', 'prompt_name']:
        if trait in cluster_data.columns:
            trait_dist = cluster_data[trait].value_counts()
            print(f"\n{trait.upper()}:")
            for value, count in trait_dist.items():
                percentage = count / len(cluster_data) * 100
                print(f"  {value:30s}: {count:4d} ({percentage:5.1f}%)")

## Etapa 12: Análise Qualitativa - Textos Representativos

Examinar textos representativos de cada cluster.

In [ ]:
def get_cluster_representative_texts(df, cluster_col, n_samples=3):
    """
    Obtém textos representativos de cada cluster.
    """
    print("="*80)
    print("TEXTOS REPRESENTATIVOS POR CLUSTER")
    print("="*80)
    
    for cluster_id in sorted(df[cluster_col].unique()):
        if cluster_id == -1:
            continue
        
        cluster_data = df[df[cluster_col] == cluster_id]
        
        print(f"\n{'─'*80}")
        print(f"CLUSTER {cluster_id}")
        print(f"{'─'*80}")
        
        # Pegar amostras aleatórias
        samples = cluster_data.sample(min(n_samples, len(cluster_data)), random_state=42)
        
        for idx, (_, row) in enumerate(samples.iterrows(), 1):
            print(f"\nExemplo {idx}:")
            print(f"  Gênero: {row.get('genero', 'N/A')}, Raça: {row.get('raca', 'N/A')}, Região: {row.get('regiao', 'N/A')}")
            print(f"  Modelo: {row.get('model', 'N/A')}")
            if 'response' in row:
                text = row['response'][:200] + '...' if len(str(row['response'])) > 200 else row['response']
                print(f"  Texto: {text}")

# Mostrar textos representativos
get_cluster_representative_texts(merged_df, cluster_col, n_samples=2)

## Resumo e Conclusões

### Principais Achados:

1. **Melhor Algoritmo**: Identificado através da análise de pureza
2. **Alinhamento com Traços**: Verificar se clusters se alinham melhor com persona traits ou com aspectos técnicos (modelo/prompt)
3. **Padrões Semânticos**: Clusters revelam "mundos semânticos" diferentes?

### Interpretação:

- **Alta pureza em 'model'**: Modelos diferentes criam representações distintas
- **Alta pureza em traços de persona**: Embeddings capturam diferenças entre personas
- **Baixa pureza geral**: Embeddings são complexos e multidimensionais

### Próximos Passos:

1. Refinar número de clusters
2. Testar outros hiperparâmetros (eps para DBSCAN, etc.)
3. Aplicar clustering hierárquico e visualizar dendrogramas
4. Análise de co-ocorrência entre traços
5. Integrar com análise de métricas textuais

In [ ]:
# Salvar resultados
output_path = '/home/igo/faculdade/poc/embeddings/clustering_results.parquet'
merged_df.to_parquet(output_path, index=False)
print(f"\n✓ Resultados salvos em: {output_path}")

print("\n" + "="*80)
print("ANÁLISE DE CLUSTERING CONCLUÍDA!")
print("="*80)